# Packages

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data/A0008D - v_pt_ride_all (full).csv')

In [3]:
df['BIZ_DT'] = pd.to_datetime(df['BIZ_DT'])
df['ENTRY_DT'] = pd.to_datetime(df['ENTRY_DT'])
df['EXIT_DT'] = pd.to_datetime(df['EXIT_DT'])
df['ENTRY_TM'] = pd.to_datetime(
    df['ENTRY_DT'].astype(str) + ' ' + df['ENTRY_TM'],
    format='%Y-%m-%d %H:%M:%S.%f'
)
df['EXIT_TM'] = pd.to_datetime(
    df['EXIT_DT'].astype(str) + ' ' + df['EXIT_TM'],
    format='%Y-%m-%d %H:%M:%S.%f'
)

In [4]:
df.dtypes

BIZ_DT                datetime64[us]
BUS_SVC_NUM                  float64
CRD_NUM                        int64
DEST_LOC_ID_NUM                int64
ENTRY_DT              datetime64[us]
ENTRY_TM              datetime64[us]
EXIT_DT               datetime64[us]
EXIT_TM               datetime64[us]
JRNY_ID_NUM                    int64
ORIG_LOC_ID_NUM                int64
PATRON_CATG_ID_NUM             int64
PAY_CD                         int64
RIDE_DISC_AMT                float64
RIDE_DIST_KM_CNT             float64
RIDE_FARE_AMT                float64
RIDE_ID_NUM                    int64
RIDE_MIN_CNT                 float64
RIDE_ST_CD                     int64
SVC_GRADE_ID_NUM               int64
TKT_TYP_CD                     int64
TRNSPT_MODE_CD                 int64
dtype: object

In [5]:
df1 = df[df['ENTRY_DT'] == '2025-02-11'].copy()
df1.columns
df2 = df1[['BUS_SVC_NUM', 'CRD_NUM', 'DEST_LOC_ID_NUM', 'ENTRY_DT',
       'ENTRY_TM', 'EXIT_DT', 'EXIT_TM', 'ORIG_LOC_ID_NUM', 'RIDE_DISC_AMT', 'RIDE_DIST_KM_CNT',
       'RIDE_FARE_AMT', 'RIDE_ID_NUM', 'RIDE_MIN_CNT']]
df2.head()

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,RIDE_DIST_KM_CNT,RIDE_FARE_AMT,RIDE_ID_NUM,RIDE_MIN_CNT
58167,62.0,200003194601,6492,2025-02-11,2025-02-11 00:47:26,2025-02-11,2025-02-11 01:13:17,3979,0.0,9.1,1.73,110848327669,25.850
59621,117.0,230021568683,6034,2025-02-11,2025-02-11 00:23:54,2025-02-11,2025-02-11 00:29:20,6042,0.0,1.1,1.19,110848360426,5.433
60686,NaN,230036729632,325,2025-02-11,2025-02-11 00:03:46,2025-02-11,2025-02-11 00:18:29,320,0.0,6.1,1.50,110848362684,14.717
61474,235.0,2190006112681,3380,2025-02-11,2025-02-11 00:33:00,2025-02-11,2025-02-11 00:36:29,6984,0.0,0.8,0.00,110848329634,3.483
61528,979.0,190002646194,2997,2025-02-11,2025-02-11 00:01:31,2025-02-11,2025-02-11 00:29:25,6901,0.0,6.7,0.24,110848330026,27.900


In [6]:
bus_data = pd.read_csv('../../data/A0008D - v_q_bus_stop (full).csv')
bus_vls_data = pd.read_csv('../../data/A0008D - v_q_vls_marker (full).csv')

In [7]:
bus_consolidated = bus_data.merge(bus_vls_data, how= 'left', left_on= 'MRK_ID_NUM',
                                  right_on='MRK_ID_NUM')
bus_needed = bus_consolidated[['BUS_STOP_NAM', 'MRK_ID_NUM', 'LATITUDE_VAL', 'LONGITUDE_VAL']].copy()
bus_needed = bus_needed.rename(columns={
    'LATITUDE_VAL': 'LATITUDE',
    'LONGITUDE_VAL': 'LONGITUDE',
    'BUS_STOP_NAM': 'STATION_NAME'
})
bus_needed.head(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE
0,Opp Mandai Rd,2940,5079920,373520243
1,SAF FERRY TERMINAL,6367,4996948,374397856
2,Loyang Court,6981,0,0
3,Os Jurong Frog Farm,6251,0,0
4,JURONG DEPOT,1771,4767738,373451957


In [8]:
train_data = pd.read_csv('../../data/mrt_station_coordinates.csv')
train_needed = train_data[['NODE_NAME', 'NODE_MRK_ID', 'LATITUDE', 'LONGITUDE']]
train_needed = train_needed.rename(columns={
    'NODE_NAME': 'STATION_NAME',
    'NODE_MRK_ID': 'MRK_ID_NUM'
})
train_needed.head(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE
0,Yio Chu Kang,1,1.381756,103.844947
1,Ang Mo Kio,2,1.369933,103.849558
2,Bishan NSEW,3,1.350839,103.848144
3,Braddell,4,1.340469,103.846799
4,Toa Payoh,5,1.332629,103.847502


In [9]:
consolidated_stations = pd.concat([bus_needed, train_needed], axis = 0)
consolidated_stations.tail(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE
201,Marine Parade,427,1.302865,103.905508
202,Marine Terrace,428,1.306786,103.915316
203,Siglap,429,1.309877,103.929879
204,Bayshore,430,1.312841,103.941597
205,Hume,336,1.354511,103.769104


In [10]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
                right_on= 'MRK_ID_NUM')
df2.head(5)

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,RIDE_DIST_KM_CNT,RIDE_FARE_AMT,RIDE_ID_NUM,RIDE_MIN_CNT,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE
0,62.0,200003194601,6492,2025-02-11,2025-02-11 00:47:26,2025-02-11,2025-02-11 01:13:17,3979,0.0,9.1,1.73,110848327669,25.850,Coral Edge Stn Exit A,6492.0,5.019504e+06,3.740830e+08
1,117.0,230021568683,6034,2025-02-11,2025-02-11 00:23:54,2025-02-11,2025-02-11 00:29:20,6042,0.0,1.1,1.19,110848360426,5.433,Opp Blk 104B,6034.0,5.212670e+06,3.737894e+08
2,NaN,230036729632,325,2025-02-11,2025-02-11 00:03:46,2025-02-11,2025-02-11 00:18:29,320,0.0,6.1,1.50,110848362684,14.717,Ubi,325.0,1.329974e+00,1.038992e+02
3,235.0,2190006112681,3380,2025-02-11,2025-02-11 00:33:00,2025-02-11,2025-02-11 00:36:29,6984,0.0,0.8,0.00,110848329634,3.483,Braddell Stn/Blk 111,3380.0,4.827108e+06,3.738436e+08
4,979.0,190002646194,2997,2025-02-11,2025-02-11 00:01:31,2025-02-11,2025-02-11 00:29:25,6901,0.0,6.7,0.24,110848330026,27.900,Regent Gr Condo,2997.0,5.039427e+06,3.734997e+08


In [11]:
df2 = df2.rename(columns={
    'STATION_NAME': 'DEST_STATION_NAME',
    'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
    'LATITUDE': 'DEST_LATITUDE',
    'LONGITUDE': 'DEST_LONGITUDE'
})

In [12]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
                right_on= 'MRK_ID_NUM')

In [13]:
df2 = df2.rename(columns={
    'STATION_NAME': 'ORIG_STATION_NAME',
    'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
    'LATITUDE': 'ORIG_LATITUDE',
    'LONGITUDE': 'ORIG_LONGITUDE'
})
df2.head(5)

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,RIDE_DIST_KM_CNT,...,RIDE_ID_NUM,RIDE_MIN_CNT,DEST_STATION_NAME,DEST_MRK_ID_NUM,DEST_LATITUDE,DEST_LONGITUDE,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE
0,62.0,200003194601,6492,2025-02-11,2025-02-11 00:47:26,2025-02-11,2025-02-11 01:13:17,3979,0.0,9.1,...,110848327669,25.850,Coral Edge Stn Exit A,6492.0,5.019504e+06,3.740830e+08,The Minton,3979.0,4.863864e+06,3.739758e+08
1,117.0,230021568683,6034,2025-02-11,2025-02-11 00:23:54,2025-02-11,2025-02-11 00:29:20,6042,0.0,1.1,...,110848360426,5.433,Opp Blk 104B,6034.0,5.212670e+06,3.737894e+08,Canberra Stn,6042.0,5.193281e+06,3.737885e+08
2,NaN,230036729632,325,2025-02-11,2025-02-11 00:03:46,2025-02-11,2025-02-11 00:18:29,320,0.0,6.1,...,110848362684,14.717,Ubi,325.0,1.329974e+00,1.038992e+02,Jalan Besar,320.0,1.305171e+00,1.038553e+02
3,235.0,2190006112681,3380,2025-02-11,2025-02-11 00:33:00,2025-02-11,2025-02-11 00:36:29,6984,0.0,0.8,...,110848329634,3.483,Braddell Stn/Blk 111,3380.0,4.827108e+06,3.738436e+08,Caldecott Stn Exit 1,6984.0,4.811407e+06,3.738227e+08
4,979.0,190002646194,2997,2025-02-11,2025-02-11 00:01:31,2025-02-11,2025-02-11 00:29:25,6901,0.0,6.7,...,110848330026,27.900,Regent Gr Condo,2997.0,5.039427e+06,3.734997e+08,Bukit Panjang Interchange B3,6901.0,4.963266e+06,3.735460e+08
